[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C50_HuggingFace_Ecosystem_Course/05_accelerate_hub_api/05_accelerate_hub_api.ipynb)

# 05 · Accelerate、Hub 与推理 API（迷你复刻分布式切分 / 权重键名 / 重试限流客户端）

目标：把 **prepare(dataloader) 的分布式切分 → gather_for_metrics 的去重 → unwrap 前后的键名 →
revision 与缓存布局 → 指数退避+抖动 → RPM/TPM 双限 → 幂等 → 流式增量拼接 → 上下文预算** 从零写一遍。

路线：分布式切分与有效 batch → gather 的重复样本问题 → unwrap 与 module. 前缀 →
safetensors vs pickle → 退避与惊群 → 双限令牌桶 → 幂等缓存 → 流式拼接与 usage →
上下文预算与滑窗 → ✏️ 练习 → 📖 答案 → 🧪 成本可观测胶囊。

> 心智模型：**这一层的技术含量全在「失败处理」上**——少任何一个防护，系统就会在你没预料到的时刻出问题。

## 1 · prepare(dataloader)：换掉 sampler、每个 rank 只拿 1/N

**你的 `batch_size` 是每个进程的** —— 8 卡时有效 batch 是 8 倍。
这与 `TrainingArguments.per_device_train_batch_size` 是同一件事（模块 03 的乘法关系）。

In [ ]:
import numpy as np, math, json, hashlib, random, time
rng = np.random.default_rng(0)

def distributed_sampler(n_samples, rank, world_size, drop_last=False, seed=0, epoch=0):
    '''复刻 DistributedSampler：按 rank 交错切分；不整除时**重复补齐**（关键！）。'''
    idx = list(np.random.default_rng(seed + epoch).permutation(n_samples))
    if drop_last:
        usable = (n_samples // world_size) * world_size
        idx = idx[:usable]
    else:
        per = math.ceil(n_samples / world_size)
        total = per * world_size
        # 为了让每个 rank 拿到相同数量，**在末尾重复一些样本**
        while len(idx) < total:
            idx += idx[:total - len(idx)]
    return idx[rank::world_size]

N, W = 10, 4
parts = [distributed_sampler(N, r, W, seed=1) for r in range(W)]
print(f'{N} 个样本切给 {W} 个 rank（不 drop_last）:')
for r, p in enumerate(parts):
    print(f'  rank {r}: {p}  (len={len(p)})')
sizes = [len(p) for p in parts]
assert len(set(sizes)) == 1, '每个 rank 必须拿到相同数量（否则 all-reduce 会卡死）'
flat = [i for p in parts for i in p]
assert len(flat) == sizes[0] * W
n_dup = len(flat) - len(set(flat))
print(f'\n⚠️  为了整除，末尾**重复了 {n_dup} 个样本** —— 这是 gather_for_metrics 要处理的问题。')
assert n_dup > 0, '10 不能被 4 整除 -> 必然有重复'

parts_drop = [distributed_sampler(N, r, W, drop_last=True, seed=1) for r in range(W)]
flat_drop = [i for p in parts_drop for i in p]
assert len(flat_drop) == len(set(flat_drop)), 'drop_last=True 无重复'
assert len(flat_drop) < N, '但会丢掉最后几个样本'
print(f'drop_last=True: 无重复，但丢了 {N - len(flat_drop)} 个样本（训练可接受，评估不可）')

In [ ]:
def effective_batch(per_device_bs, grad_accum, world_size):
    return per_device_bs * grad_accum * world_size

print(f"{'per_device':>11s} {'accum':>6s} {'world':>6s} {'有效batch':>10s}")
for pd, ga, ws in [(8, 4, 1), (8, 4, 8), (8, 1, 8), (1, 32, 8)]:
    print(f'{pd:>11d} {ga:>6d} {ws:>6d} {effective_batch(pd,ga,ws):>10d}')
assert effective_batch(8, 4, 1) == 32 and effective_batch(8, 4, 8) == 256
print('\n⚠️  同一份代码、同样的 batch_size，1 卡与 8 卡的有效 batch 差 8 倍。')
print('✅ 换机器时必须重新算有效 batch —— 这是「换台机器超参全变」的根源。')

# 累积期间跳过梯度同步能省多少通信
def grad_comm_volume(n_params, grad_accum, world_size, skip_intermediate=True):
    '''ring all-reduce 的通信量 ≈ 2*(W-1)/W * N 每次同步。'''
    per_sync = 2 * (world_size - 1) / world_size * n_params
    n_syncs = 1 if skip_intermediate else grad_accum
    return per_sync * n_syncs

N_P, GA, WS = 7e9, 8, 8
naive = grad_comm_volume(N_P, GA, WS, skip_intermediate=False)
smart = grad_comm_volume(N_P, GA, WS, skip_intermediate=True)
print(f'\ngrad_accum={GA}, {WS} 卡, 7B 参数:')
print(f'  每步都同步 : {naive*2/1e9:>8.1f} GB/优化步 (bf16)')
print(f'  只在最后同步: {smart*2/1e9:>8.1f} GB/优化步')
print(f'  省 {(1-smart/naive):.0%} 的梯度通信量')
assert abs(naive / smart - GA) < 1e-9
print('✅ accelerator.accumulate() 自动用 no_sync() 做这件事 —— 自己写 DDP 循环常常漏掉。')

## 2 · gather_for_metrics：为什么裸 gather 会让指标有偏

In [ ]:
def naive_gather(per_rank_preds):
    '''裸 gather：把所有 rank 的预测拼起来 —— **包含重复样本**。'''
    return [x for p in per_rank_preds for x in p]

def gather_for_metrics(per_rank_preds, n_original):
    '''知道原始长度，裁掉为整除而重复的部分。'''
    return naive_gather(per_rank_preds)[:n_original]

# 20 个样本、6 个 rank（不整除）
N_EVAL, WS2 = 20, 6
labels = list(rng.integers(0, 2, size=N_EVAL))
parts = [distributed_sampler(N_EVAL, r, WS2, seed=2) for r in range(WS2)]
# 假设模型对前 18 个样本全对、后 2 个全错（构造一个可验证的场景）
def predict(i): return labels[i] if i < 18 else 1 - labels[i]
per_rank = [[(i, predict(i)) for i in p] for p in parts]

def accuracy(pairs):
    return float(np.mean([pred == labels[i] for i, pred in pairs]))

g_naive = naive_gather(per_rank)
# 用「原始索引集合」去重来模拟正确做法
seen, g_dedup = set(), []
for i, pred in g_naive:
    if i not in seen:
        seen.add(i); g_dedup.append((i, pred))

print(f'原始样本数 {N_EVAL}, {WS2} 个 rank')
print(f'裸 gather 收到 {len(g_naive)} 条（含 {len(g_naive)-len(g_dedup)} 条重复）')
print(f'去重后        {len(g_dedup)} 条')
print(f'\naccuracy: 裸 gather {accuracy(g_naive):.4f} | 去重后 {accuracy(g_dedup):.4f}')
assert len(g_naive) > N_EVAL, '裸 gather 的条数超过原始样本数'
assert len(g_dedup) == N_EVAL
assert accuracy(g_naive) != accuracy(g_dedup), '重复样本会让指标有偏'
print('\n⚠️  症状：「多卡评估结果与单卡略有不同」—— 很容易被误认为是随机性。')
print('✅ 用 accelerator.gather_for_metrics()，它知道原始长度并裁掉重复部分。')

## 3 · unwrap_model：`module.` 前缀会让所有键 missing

In [ ]:
class DDPWrapper:
    '''DDP 包装：state_dict 的键名会多一个 module. 前缀。'''
    def __init__(self, module): self.module = module
    def state_dict(self):
        return {f'module.{k}': v for k, v in self.module.state_dict().items()}

class TinyNet:
    def __init__(self):
        self._sd = {'encoder.weight': np.ones((4, 4)), 'classifier.weight': np.zeros((4, 2))}
    def state_dict(self): return dict(self._sd)

net = TinyNet()
wrapped = DDPWrapper(net)
print('未包装的键:', sorted(net.state_dict()))
print('DDP 包装后:', sorted(wrapped.state_dict()))

def load_into(model_keys, state_dict):
    file_keys = set(state_dict)
    return {'missing': sorted(set(model_keys) - file_keys),
            'unexpected': sorted(file_keys - set(model_keys))}

model_keys = list(net.state_dict())
bad = load_into(model_keys, wrapped.state_dict())
good = load_into(model_keys, net.state_dict())
print(f'\n❌ 直接保存 DDP 包装后的模型: missing={bad["missing"]}')
print(f'                              unexpected={bad["unexpected"]}')
print(f'✅ 先 unwrap 再保存:            missing={good["missing"]}, unexpected={good["unexpected"]}')
assert len(bad['missing']) == len(model_keys), '**全部键都 missing** -> 模型等于随机初始化'
assert good['missing'] == [] and good['unexpected'] == []
print('\n⚠️  这正是模块 01 那个坑：加载「成功」了，只给个警告，但模型是随机的。')
print('✅ accelerator.unwrap_model(model) 之后再保存。')

def strip_prefix(sd, prefix='module.'):
    '''补救：手动剥前缀（拿到别人错存的 checkpoint 时用）。'''
    return {(k[len(prefix):] if k.startswith(prefix) else k): v for k, v in sd.items()}
fixed = load_into(model_keys, strip_prefix(wrapped.state_dict()))
assert fixed['missing'] == [] and fixed['unexpected'] == []
print('✅ 补救手段：strip_prefix("module.") 能修好已经错存的 checkpoint')

## 4 · safetensors vs pickle：格式决定安全边界

In [ ]:
# 用一个「假 pickle」演示：反序列化时会执行 __reduce__ 里的东西
EXECUTED = []

class MaliciousPayload:
    '''模拟恶意 pickle：反序列化时执行任意代码。'''
    def __reduce__(self):
        return (EXECUTED.append, ('攻击者的代码被执行了',))

def fake_pickle_load(obj):
    '''模拟 pickle.load：会调用 __reduce__ 并执行。'''
    if hasattr(obj, '__reduce__') and not isinstance(obj, (dict, list, np.ndarray)):
        fn, args = obj.__reduce__()
        return fn(*args)
    return obj

def safetensors_load(header_and_bytes):
    '''模拟 safetensors：纯数据解析，**不会调用任何用户代码**。'''
    header, blob = header_and_bytes
    out = {}
    for name, meta in header.items():
        s, e = meta['offsets']
        out[name] = np.frombuffer(blob[s:e], dtype=meta['dtype']).reshape(meta['shape'])
    return out

assert EXECUTED == []
fake_pickle_load(MaliciousPayload())
print(f'❌ pickle 路径: EXECUTED = {EXECUTED}   ← 加载权重就执行了代码')
assert EXECUTED == ['攻击者的代码被执行了'], 'pickle 的反序列化会执行 __reduce__'

# safetensors：header 是纯 JSON，数据是连续字节，没有执行入口
arr = np.arange(8, dtype=np.float32)
blob = arr.tobytes()
header = {'weight': {'dtype': 'float32', 'shape': [2, 4], 'offsets': [0, len(blob)]}}
EXECUTED.clear()
loaded = safetensors_load((header, blob))
print(f'\n✅ safetensors 路径: EXECUTED = {EXECUTED}   ← 纯数据，无执行入口')
assert EXECUTED == [], 'safetensors 不执行任何用户代码'
assert np.allclose(loaded['weight'], arr.reshape(2, 4))
# 且支持按名字部分加载 / mmap（零拷贝）
assert list(header) == ['weight'], 'header 里有全部张量的名字与偏移 -> 可单独读某个张量'
print('   而且 header 里有全部张量的名字与偏移 -> 可按需部分加载、可 mmap 零拷贝。')
print('\n✅ 生产规则：强制 use_safetensors=True；trust_remote_code 只对可信 repo 开。')

### revision：钉死 commit sha 才可复现

In [ ]:
HUB = {                       # 模拟 Hub：一个 repo 的历史
    'org/model': {
        'main': 'sha_c3',                                   # 可变标签！
        'v1.0': 'sha_a1',
        'commits': {'sha_a1': {'w': 1.0}, 'sha_b2': {'w': 2.0}, 'sha_c3': {'w': 3.0}},
    }
}
CACHE = {}

def hub_download(repo, revision='main'):
    '''缓存按 **commit sha** 分目录 -> 钉死 revision 既可复现也让缓存稳定命中。'''
    meta = HUB[repo]
    sha = meta.get(revision, revision)
    if sha not in meta['commits']:
        raise FileNotFoundError(f'revision {revision!r} not found')
    path = f'~/.cache/huggingface/hub/models--{repo.replace("/","--")}/snapshots/{sha}/'
    hit = path in CACHE
    CACHE[path] = meta['commits'][sha]
    return meta['commits'][sha], sha, path, hit

w1, sha1, p1, _ = hub_download('org/model', 'main')
print(f'revision="main"  -> sha={sha1}, w={w1["w"]}')
HUB['org/model']['main'] = 'sha_b2'                 # 上游更新了 main
w2, sha2, p2, _ = hub_download('org/model', 'main')
print(f'上游更新 main 后 -> sha={sha2}, w={w2["w"]}   ← **权重悄悄变了**')
assert w1['w'] != w2['w'], '用 main 会在上游更新后静默换模型'

w3, sha3, _, _ = hub_download('org/model', 'sha_a1')
HUB['org/model']['main'] = 'sha_c3'
w4, sha4, _, hit = hub_download('org/model', 'sha_a1')
print(f'\nrevision="sha_a1" -> w={w3["w"]}；上游再变后仍然 w={w4["w"]}，缓存命中={hit}')
assert w3 == w4 and sha3 == sha4, '钉死 sha 后完全可复现'
assert hit, '同一个 sha 命中同一个缓存目录'
print('\n⚠️  用 main 的项目在上游更新权重后会静默换模型 ——')
print('   这是「昨天还好今天变差」的一个真实来源。')
print('✅ 生产上钉 commit sha，并把它记进实验日志（C37/C40）。')

## 5 · 客户端四层防护：退避+抖动 / 双限 / 幂等 / 超时

In [ ]:
RETRYABLE = {408, 429, 500, 502, 503, 504}
NON_RETRYABLE = {400, 401, 403, 404, 422}

class APIError(Exception):
    def __init__(self, status, retry_after=None):
        super().__init__(f'HTTP {status}')
        self.status, self.retry_after = status, retry_after

def backoff_delay(attempt, base=0.5, cap=30.0, jitter=True, rnd=None):
    '''指数退避 + 抖动。抖动**不是可选的** —— 它防惊群。'''
    d = min(cap, base * (2 ** attempt))
    if jitter:
        r = rnd or random.Random()
        d *= r.uniform(0.5, 1.5)
    return d

def should_retry(err):
    return isinstance(err, APIError) and err.status in RETRYABLE

def call_with_retry(fn, max_attempts=5, base=0.5, cap=30.0, rnd=None, sleep=lambda s: None):
    delays = []
    for attempt in range(max_attempts):
        try:
            return fn(attempt), delays
        except APIError as e:
            if not should_retry(e) or attempt == max_attempts - 1:
                raise
            # 优先尊重 Retry-After
            d = e.retry_after if e.retry_after is not None else \
                backoff_delay(attempt, base, cap, rnd=rnd)
            delays.append(round(d, 3)); sleep(d)
    raise RuntimeError('unreachable')

# 前 3 次 429，第 4 次成功
calls = {'n': 0}
def flaky(attempt):
    calls['n'] += 1
    if calls['n'] <= 3: raise APIError(429)
    return {'ok': True}

res, delays = call_with_retry(flaky, rnd=random.Random(0))
print(f'成功（第 {calls["n"]} 次尝试），退避序列: {delays}')
assert res['ok'] and len(delays) == 3
assert delays[1] > delays[0] * 1.2, '退避应大致指数增长'

# 400 不重试：重试一万次也一样错
def bad_request(attempt): raise APIError(400)
try:
    call_with_retry(bad_request)
    raise RuntimeError('不该到这')
except APIError as e:
    assert e.status == 400
    print(f'✅ HTTP {e.status} 不重试（请求本身有问题，必须报出来）')

# Retry-After 优先
def rate_limited(attempt):
    if attempt == 0: raise APIError(429, retry_after=7.5)
    return {'ok': True}
_, d2 = call_with_retry(rate_limited, rnd=random.Random(0))
assert d2 == [7.5], f'应尊重 Retry-After，得到 {d2}'
print(f'✅ 尊重 Retry-After: {d2}')

In [ ]:
# 抖动为什么必需：模拟 200 个客户端同时遇到 429
def herd_peak(n_clients, jitter, bucket=0.25, attempts=3):
    '''统计重试时刻的分布：峰值越高说明惊群越严重。'''
    times = []
    for c in range(n_clients):
        rnd = random.Random(c); t = 0.0
        for a in range(attempts):
            t += backoff_delay(a, jitter=jitter, rnd=rnd)
            times.append(t)
    hist = {}
    for t in times:
        hist[round(t / bucket)] = hist.get(round(t / bucket), 0) + 1
    return max(hist.values()), len(times)

peak_no, tot = herd_peak(200, jitter=False)
peak_yes, _ = herd_peak(200, jitter=True)
print(f'200 客户端 × 3 次重试 = {tot} 次请求')
print(f'  无抖动: 单个 0.25s 窗口内峰值 {peak_no} 次请求')
print(f'  有抖动: 单个 0.25s 窗口内峰值 {peak_yes} 次请求')
assert peak_no > peak_yes * 2, '无抖动时所有客户端同步重试 -> 周期性洪峰'
print(f'\n✅ 抖动把峰值降低 {peak_no/peak_yes:.1f}× —— 这是分布式系统里最便宜的一行代码。')

In [ ]:
class TokenBucket:
    def __init__(self, rate, capacity):
        self.rate, self.cap, self.tokens, self.last = rate, float(capacity), float(capacity), 0.0
    def allow(self, now, cost=1.0):
        self.tokens = min(self.cap, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= cost:
            self.tokens -= cost; return True
        return False
    def refund(self, amount):
        self.tokens = min(self.cap, self.tokens + max(0.0, amount))

class RateLimitedClient:
    '''RPM + TPM 双限 + 预扣/结算 + 幂等缓存。'''
    def __init__(self, rpm, tpm, idem_window=300.0):
        self.rpm_b = TokenBucket(rpm / 60.0, rpm)
        self.tpm_b = TokenBucket(tpm / 60.0, tpm)
        self.idem, self.window = {}, idem_window
        self.stats = {'served': 0, 'throttled': 0, 'idem_hits': 0, 'tokens': 0}
    def call(self, now, est_tokens, actual_tokens=None, idem_key=None):
        if idem_key is not None and idem_key in self.idem:
            t0, resp = self.idem[idem_key]
            if now - t0 <= self.window:
                self.stats['idem_hits'] += 1
                return resp                     # 幂等命中：不重复消耗配额、不重复计费
        if not self.rpm_b.allow(now):
            self.stats['throttled'] += 1; return {'error': 429, 'reason': 'RPM'}
        if not self.tpm_b.allow(now, cost=est_tokens):
            self.rpm_b.refund(1); self.stats['throttled'] += 1
            return {'error': 429, 'reason': 'TPM'}
        used = est_tokens if actual_tokens is None else actual_tokens
        self.tpm_b.refund(est_tokens - used)     # 结算：退还多扣的
        self.stats['served'] += 1; self.stats['tokens'] += used
        resp = {'ok': True, 'usage': {'total_tokens': used}}
        if idem_key is not None: self.idem[idem_key] = (now, resp)
        return resp

def max_throughput(rpm, tpm, avg_tokens):
    return min(rpm, tpm // avg_tokens)

print(f"{'RPM':>6s} {'TPM':>8s} {'均token':>8s} {'RPM上限':>8s} {'TPM上限':>8s} {'实际瓶颈':>10s}")
for rpm, tpm, avg in [(60, 60_000, 500), (60, 60_000, 2000), (3500, 90_000, 1500)]:
    lim_r, lim_t = rpm, tpm // avg
    print(f'{rpm:>6d} {tpm:>8d} {avg:>8d} {lim_r:>8d} {lim_t:>8d} '
          f'{("RPM" if lim_r < lim_t else "TPM"):>10s}')
assert max_throughput(3500, 90_000, 1500) == 60, '看起来 RPM=3500，实际只能 60 请求/分'
print('\n⚠️  「RPM=3500」看起来能每秒 58 个请求，实际 TPM 只允许 60 个/分钟 —— 差 58 倍。')
print('✅ 大多数时候真正的瓶颈是 **TPM 而不是 RPM**。')

In [ ]:
c = RateLimitedClient(rpm=60, tpm=60_000)
# 长生成请求：预扣 4000、实际只用 500 -> 结算退还
r1 = c.call(0.0, est_tokens=4000, actual_tokens=500, idem_key='req-1')
assert r1['ok'] and r1['usage']['total_tokens'] == 500
# 同一个幂等键重试：不重复消耗配额
before = dict(c.stats)
r2 = c.call(1.0, est_tokens=4000, actual_tokens=500, idem_key='req-1')
assert r2 is r1 or r2 == r1
assert c.stats['served'] == before['served'], '幂等命中不应再消耗配额'
assert c.stats['idem_hits'] == 1
print(f'幂等: 第二次调用命中缓存，served 仍为 {c.stats["served"]}，idem_hits={c.stats["idem_hits"]}')

# 打爆 TPM
c2 = RateLimitedClient(rpm=1000, tpm=6000)
oks = sum(1 for i in range(10) if c2.call(0.0, est_tokens=1000).get('ok'))
print(f'TPM=6000, 每请求 1000 token: 前 10 个请求放行 {oks} 个')
assert oks == 6, 'TPM 桶容量 6000 -> 只放行 6 个'
assert c2.stats['throttled'] == 4
print('✅ 双限 + 预扣结算 + 幂等 三件套工作正常')

## 6 · 流式：增量拼接、usage 与断连

In [ ]:
def fake_stream(text, include_usage=False):
    '''模拟 OpenAI 流式响应。注意第一个 chunk 只有 role、content 为 None。'''
    yield {'choices': [{'delta': {'role': 'assistant'}}]}
    for ch in text:
        yield {'choices': [{'delta': {'content': ch}}]}
    yield {'choices': [{'delta': {}, 'finish_reason': 'stop'}]}
    if include_usage:
        yield {'choices': [], 'usage': {'prompt_tokens': 12, 'completion_tokens': len(text),
                                        'total_tokens': 12 + len(text)}}

def consume_stream(stream, cancel_after=None):
    '''正确的增量拼接：跳过 content 为 None 的 chunk；记录 usage；支持取消。'''
    parts, usage, finish, n = [], None, None, 0
    for chunk in stream:
        n += 1
        if chunk.get('usage'):
            usage = chunk['usage']
        for ch in chunk.get('choices', []):
            d = ch.get('delta', {})
            if d.get('content') is not None:      # ← 必须判 None，不能直接取
                parts.append(d['content'])
            if ch.get('finish_reason'):
                finish = ch['finish_reason']
        if cancel_after is not None and n >= cancel_after:
            return ''.join(parts), usage, 'cancelled', n
    return ''.join(parts), usage, finish, n

text, usage, finish, n = consume_stream(fake_stream('LoRA 很好'))
print(f'拼接结果: {text!r}, finish={finish}, chunks={n}')
print(f'usage: {usage}   ← 默认流式**不带 usage**！')
assert text == 'LoRA 很好' and finish == 'stop'
assert usage is None, '默认流式不返回 usage -> 无法回答「这次花了多少 token」'

text2, usage2, _, _ = consume_stream(fake_stream('LoRA 很好', include_usage=True))
assert usage2 is not None and usage2['completion_tokens'] == len('LoRA 很好')
print(f'\n开了 stream_options={{"include_usage": True}} 后: usage={usage2} ✅')

# 断连要真正取消：否则服务端还在为你生成
t3, _, f3, n3 = consume_stream(fake_stream('这是一段很长的回答' * 5), cancel_after=6)
print(f'\n提前取消: 只拿到 {len(t3)} 个字符（{n3} 个 chunk），finish={f3}')
assert f3 == 'cancelled' and n3 == 6
print('⚠️  客户端放弃后必须真正关闭连接（用 with / close()），')
print('   否则服务端还在为你生成 —— 既烧你的配额也烧服务端算力。')
print('✅ 三件事：判 None 再拼接、显式开 include_usage、断连要取消。')

## 7 · 上下文预算：多轮对话会越来越紧

$$L_{avail}(t) = L_{model} - L_{sys} - \sum_{i<t}(L_{user,i}+L_{assist,i}) - L_{reserve}$$

In [ ]:
def context_state(model_max, sys_tokens, history, reserve):
    used = sys_tokens + sum(history)
    return {'used': used, 'available': model_max - used - reserve,
            'fits': model_max - used - reserve > 0}

MODEL_MAX, SYS, RESERVE = 4096, 200, 512
history = []
print(f'model_max={MODEL_MAX}, system={SYS}, 生成预留={RESERVE}')
print(f"{'轮次':>4s} {'历史token':>10s} {'可用':>8s} {'状态':>8s}")
for turn in range(1, 12):
    history += [180, 320]                     # 每轮 user 180 + assistant 320
    s = context_state(MODEL_MAX, SYS, history, RESERVE)
    if turn % 2 == 1 or not s['fits']:
        print(f'{turn:>4d} {sum(history):>10d} {s["available"]:>8d} '
              f'{("✅" if s["fits"] else "❌ 超限"):>8s}')
    if not s['fits']: break
assert not context_state(MODEL_MAX, SYS, history, RESERVE)['fits'], '多轮后必然超限'

def sliding_window(history, model_max, sys_tokens, reserve, keep_turns=None):
    '''滑窗：从最近往前保留，直到装不下。返回保留的轮数。'''
    budget = model_max - sys_tokens - reserve
    kept, total = 0, 0
    for pair in reversed(range(0, len(history), 2)):
        cost = history[pair] + (history[pair + 1] if pair + 1 < len(history) else 0)
        if total + cost > budget: break
        total += cost; kept += 1
        if keep_turns and kept >= keep_turns: break
    return kept, total

kept, total = sliding_window(history, MODEL_MAX, SYS, RESERVE)
print(f'\n滑窗策略: 共 {len(history)//2} 轮，只能保留最近 {kept} 轮（{total} token）')
assert kept < len(history) // 2, '滑窗必然丢弃早期轮次'
assert total <= MODEL_MAX - SYS - RESERVE
print('⚠️  代价：用户会发现「它忘了前面说的话」。')
print('✅ 四种策略：滑窗（丢信息）/ 摘要压缩（额外成本+失真）/ 检索（复杂，C11-C33）/ 硬失败（诚实）。')
print('   **无论选哪种，都必须先能算出这个数字** —— 最糟的是不算，然后被静默截断。')

## ✏️ 练习 1：退避序列

实现 `retry_schedule(max_attempts, base, cap, retry_after=None, seed=0)`：
返回每次重试前的等待秒数列表（长度 `max_attempts-1`）。
若 `retry_after` 不为 None，则**全部**用它；否则用 `min(cap, base*2^k) × U(0.5,1.5)`。

In [ ]:
def retry_schedule(max_attempts, base, cap, retry_after=None, seed=0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
s = retry_schedule(5, base=0.5, cap=30.0, seed=1)
assert len(s) == 4
assert all(0.25 <= d <= 30 * 1.5 for d in s), s
assert s[3] > s[0], '整体趋势应递增'
# cap 生效
s_cap = retry_schedule(12, base=1.0, cap=8.0, seed=2)
assert max(s_cap) <= 8.0 * 1.5 + 1e-9, f'不应超过 cap*1.5，得到 {max(s_cap)}'
# retry_after 优先
assert retry_schedule(4, 0.5, 30, retry_after=7.5) == [7.5, 7.5, 7.5]
# 同 seed 可复现
assert retry_schedule(5, 0.5, 30, seed=1) == retry_schedule(5, 0.5, 30, seed=1)
print('退避序列:', [round(x, 3) for x in s])
print('✅ 练习 1 通过：**抖动不是可选的** —— 它防惊群')

## ✏️ 练习 2：双限吞吐上限

实现 `throughput_limits(rpm, tpm, avg_prompt_tokens, avg_completion_tokens)`：
返回 `{'rpm_limit':…, 'tpm_limit':…, 'bottleneck': 'RPM'|'TPM', 'requests_per_min': …}`。
注意 TPM 同时计入 prompt 与 completion。

In [ ]:
def throughput_limits(rpm, tpm, avg_prompt_tokens, avg_completion_tokens):
    # TODO: total = prompt + completion；tpm_limit = tpm // total
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
r = throughput_limits(3500, 90_000, 1200, 300)
assert r['tpm_limit'] == 60 and r['bottleneck'] == 'TPM'
assert r['requests_per_min'] == 60
r2 = throughput_limits(60, 600_000, 100, 100)
assert r2['bottleneck'] == 'RPM' and r2['requests_per_min'] == 60
print(f'RPM=3500 TPM=90k, 1200+300 token: 瓶颈={r["bottleneck"]}, 实际 {r["requests_per_min"]} 请求/分')
print(f'RPM=60 TPM=600k, 100+100 token : 瓶颈={r2["bottleneck"]}, 实际 {r2["requests_per_min"]} 请求/分')
print('✅ 练习 2 通过：容量规划前先算清哪个维度先饱和')

## ✏️ 练习 3：成本归因

实现 `cost_report(calls, price_per_1k_prompt, price_per_1k_completion)`：
`calls` 是 `[{'tag':…, 'prompt_tokens':…, 'completion_tokens':…, 'cached_prompt_tokens':…}, …]`。
缓存命中的 prompt token 按 **10%** 计价。返回 `{tag: 花费}`，外加 `'_total'` 键。

In [ ]:
def cost_report(calls, price_per_1k_prompt, price_per_1k_completion, cache_discount=0.1):
    # TODO: 每条：(未缓存prompt + 缓存prompt*cache_discount)/1000*price_p
    #             + completion/1000*price_c；按 tag 累加；加 '_total'
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
calls = [
    {'tag': 'chat',    'prompt_tokens': 1000, 'completion_tokens': 200, 'cached_prompt_tokens': 0},
    {'tag': 'chat',    'prompt_tokens': 1000, 'completion_tokens': 200, 'cached_prompt_tokens': 800},
    {'tag': 'summary', 'prompt_tokens': 5000, 'completion_tokens': 100, 'cached_prompt_tokens': 0},
]
rep = cost_report(calls, price_per_1k_prompt=0.5, price_per_1k_completion=1.5)
print({k: round(v, 5) for k, v in rep.items()})
assert '_total' in rep and abs(rep['_total'] - sum(v for k, v in rep.items() if k != '_total')) < 1e-9
# 第二条因缓存应比第一条便宜
c1 = (1000/1000)*0.5 + (200/1000)*1.5
c2 = ((200 + 800*0.1)/1000)*0.5 + (200/1000)*1.5
assert abs(rep['chat'] - (c1 + c2)) < 1e-9, f'缓存折扣算错: {rep["chat"]} vs {c1+c2}'
assert rep['summary'] > rep['chat'] / 2
print('✅ 练习 3 通过：**没有这份数据，你无法回答「哪个功能在烧钱」**')
print('   顺带一条最便宜的优化：把固定的系统提示放在 prompt 最前面 -> 命中前缀缓存')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def retry_schedule(max_attempts, base, cap, retry_after=None, seed=0):
    if retry_after is not None:
        return [retry_after] * (max_attempts - 1)
    r = random.Random(seed)
    return [min(cap, base * (2 ** k)) * r.uniform(0.5, 1.5) for k in range(max_attempts - 1)]

In [ ]:
# 练习 2 参考答案
def throughput_limits(rpm, tpm, avg_prompt_tokens, avg_completion_tokens):
    total = avg_prompt_tokens + avg_completion_tokens
    tpm_limit = tpm // total
    return {'rpm_limit': rpm, 'tpm_limit': tpm_limit,
            'bottleneck': 'RPM' if rpm <= tpm_limit else 'TPM',
            'requests_per_min': min(rpm, tpm_limit)}

In [ ]:
# 练习 3 参考答案
def cost_report(calls, price_per_1k_prompt, price_per_1k_completion, cache_discount=0.1):
    out = {}
    for c in calls:
        cached = c.get('cached_prompt_tokens', 0)
        fresh = c['prompt_tokens'] - cached
        cost = ((fresh + cached * cache_discount) / 1000) * price_per_1k_prompt \
               + (c['completion_tokens'] / 1000) * price_per_1k_completion
        out[c['tag']] = out.get(c['tag'], 0.0) + cost
    out['_total'] = sum(out.values())
    return out

---
## 🧪 真实 API 对照胶囊：一个生产可用的客户端骨架

In [ ]:
RECIPE = r'''
import os, time, random, hashlib, json
from openai import OpenAI, APIStatusError, APIConnectionError

client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"),   # 自建服务也用同一套
                api_key=os.environ["OPENAI_API_KEY"],
                timeout=60.0, max_retries=0)   # ← 关掉 SDK 重试，自己控制策略

RETRYABLE = {408, 409, 429, 500, 502, 503, 504}

def backoff(attempt, base=0.5, cap=30.0):
    return min(cap, base * 2 ** attempt) * random.uniform(0.5, 1.5)   # 抖动必需

def chat(messages, model="gpt-4o-mini", max_attempts=5,
         idem_key=None, stream=False, **kw):
    for attempt in range(max_attempts):
        try:
            extra = {}
            if idem_key:                       # ③ 幂等：重试不重复扣费
                extra["extra_headers"] = {"Idempotency-Key": idem_key}
            if stream:
                kw["stream_options"] = {"include_usage": True}   # ← 否则拿不到 usage
            return client.chat.completions.create(
                model=model, messages=messages, stream=stream, **kw, **extra)
        except APIStatusError as e:
            if e.status_code not in RETRYABLE or attempt == max_attempts - 1:
                raise                          # 400/401/404 立刻抛出，别重试
            ra = e.response.headers.get("retry-after")
            time.sleep(float(ra) if ra else backoff(attempt))    # 尊重 Retry-After
        except APIConnectionError:
            if attempt == max_attempts - 1: raise
            time.sleep(backoff(attempt))

def consume_stream(resp):
    # 正确的增量拼接：判 None、收集 usage
    parts, usage = [], None
    for chunk in resp:
        if getattr(chunk, "usage", None):
            usage = chunk.usage
        for ch in chunk.choices:
            if ch.delta.content is not None:   # ← 必须判 None
                parts.append(ch.delta.content)
    return "".join(parts), usage

# ── 结构化输出（不是 tool calling！没有函数要执行，只保证格式）──
SCHEMA = {"type": "json_schema", "json_schema": {
    "name": "verdict", "strict": True,
    "schema": {"type": "object",
               "properties": {"label": {"type": "string", "enum": ["ok", "violation"]},
                              "confidence": {"type": "number"}},
               "required": ["label", "confidence"], "additionalProperties": False}}}
r = chat([{"role": "user", "content": "审核这段话：……"}], response_format=SCHEMA)
verdict = json.loads(r.choices[0].message.content)   # schema 合法，但语义仍需校验

# ── 成本归因：每次调用都记下来（否则无法回答「哪个功能在烧钱」）──
def log_usage(tag, resp):
    u = resp.usage
    cached = getattr(getattr(u, "prompt_tokens_details", None), "cached_tokens", 0)
    print(json.dumps({"tag": tag, "model": resp.model,
                      "prompt": u.prompt_tokens, "cached": cached,
                      "completion": u.completion_tokens}, ensure_ascii=False))
'''
print(RECIPE)
for c in ['max_retries=0', 'random.uniform(0.5, 1.5)', 'Idempotency-Key',
          'include_usage', 'is not None', 'json_schema', 'cached_tokens']:
    assert c in RECIPE, c
print('✅ 配方覆盖四层防护 + 流式 + 结构化输出 + 成本归因')

### 小结
- **`prepare(dataloader)` 换掉 sampler**：`batch_size` 是每进程的，8 卡时有效 batch ×8。`accumulate()` 还会跳过中间步的梯度同步（省 7/8 通信）。
- **`gather_for_metrics` 而不是 `gather`**：分布式采样为整除会重复末尾样本，裸 gather 让指标有偏（「多卡与单卡略有不同」的真因）。
- **保存前 `unwrap_model`**：否则键名带 `module.` 前缀 → **全部 missing** → 模型等于随机初始化。
- **safetensors 不执行代码，pickle 会**（已用可运行的例子证明）；`trust_remote_code` 同理。**钉死 commit sha** 才可复现，用 `main` 会静默换模型。
- **客户端四层防护**：退避**必须加抖动**（防惊群，峰值降数倍）、**RPM+TPM 双限**（真正的瓶颈通常是 TPM）、幂等键、分层超时。
- **流式三件事**：判 `None` 再拼接、显式开 `include_usage`（默认没有 usage）、断连要真正取消。
- **上下文预算随轮数变紧**；四种策略各有代价，但**必须先能算出这个数字**。
- **记录每次调用的 token 与标签**——没有这份数据就无法回答「哪个功能在烧钱」。

🎓 **本课完结。** 你现在有了从「加载一个模型」到「训练、微调、分发、调用」的完整手艺，
以及每一层的坑清单与可复用的配方。
建议的下一站：**C48**（把服务真正部署上线）、**C39**（分布式训练的原理）、**C37**（实验追踪与生产生命周期）。